# Small experiment on event 40

In [14]:
import pandas as pd
import numpy as np
from energy_fault_detector import Config, FaultDetector
from energy_fault_detector.evaluation import Care2CompareDataset, CAREScore
from energy_fault_detector.utils.visualisation import plot_learning_curve, plot_score_with_threshold, plot_reconstruction

**Loading the data**

In [15]:
# Specify the path to the data
data_path = '../data/CARE_To_Compare'

c2c = Care2CompareDataset(path=data_path)

# event selection
event_id = 40
event_info = c2c.get_event_info(event_id=event_id)
event_info

asset_id                                    10
event_id                                    40
event_label                            anomaly
event_start                2022-12-26 00:00:00
event_start_id                           51363
event_end                  2023-01-26 13:00:00
event_end_id                             55870
event_description    Generator bearing failure
wind_farm                                    A
Name: 6, dtype: object

**Data is loaded, prepared and split into train and test data**

In [16]:
# load and split data into numerical sensor data and status_data
train_sensor_data, train_status, test_sensor_data, test_status = c2c.load_and_format_event_dataset(event_id=event_id, index_column='time_stamp')

train_sensor_data.shape, train_status.shape, test_sensor_data.shape, test_status.shape

((51219, 54), (51219,), (4939, 54), (4939,))

In [17]:
train_sensor_data.head()

,Ambient temperature (avg),Wind absolute direction (avg),Wind relative direction (avg),Windspeed (avg),Estimated windspeed (avg),Pitch angle (avg),Temperature in the hub controller (avg),Temperature in the top nacelle controller (avg),Temperature in the choke coils on the VCS-section (avg),Temperature on the VCP-board (avg),...,Active power - generator disconnected,Active power - generator connected in delta,Active power - generator connected in star,Reactive power - generator disconnected,Reactive power - generator connected in delta,Reactive power - generator connected in star,Total active power,Total reactive power,Rotor rpm (avg),Temperature in the nose cone (avg)
time_stamp,,,,,,,,,,,,,,,,,,,,,
2022-01-01 00:00:00,18.0,220.1,6.1,5.2,5.1,-1.3,27.0,38.0,93.0,39.0,...,0.0,33173.0,0.0,0.0,-13513.0,0.0,33173.0,-13513.0,11.2,19.0
2022-01-01 00:10:00,18.0,218.7,4.7,5.7,5.3,-1.6,27.0,38.0,93.0,39.0,...,0.0,38381.0,0.0,0.0,-15366.0,0.0,38381.0,-15366.0,11.7,19.0
2022-01-01 00:20:00,18.0,216.7,2.7,6.2,5.8,-1.9,27.0,38.0,93.0,39.0,...,0.0,52055.0,0.0,0.0,-15709.0,0.0,52055.0,-15709.0,12.0,19.0
2022-01-01 00:30:00,18.0,197.9,-16.1,6.3,6.2,-1.9,27.0,38.0,93.0,39.0,...,0.0,66365.0,0.0,0.0,-16444.0,0.0,66365.0,-16444.0,12.2,19.0
2022-01-01 00:40:00,18.0,217.2,3.2,6.6,6.1,-2.0,27.0,38.0,93.0,39.0,...,0.0,60980.0,0.0,0.0,-16344.0,0.0,60980.0,-16344.0,12.6,19.0


The status data contains a simple boolean value, indicating whether the status of the wind turbine was normal.

In [18]:
train_status.head()

time_stamp
2022-01-01 00:00:00    True
2022-01-01 00:10:00    True
2022-01-01 00:20:00    True
2022-01-01 00:30:00    True
2022-01-01 00:40:00    True
Name: status_type_id, dtype: bool

**Train the model**

In [20]:
# The EnergyFaultDetector needs a configuration - here we load a specific configuration for this dataset...
config = Config(config_filename='../configs/config.yaml')
# ...and update features to exclude (counters) and angles for data preprocessing
c2c.update_c2c_config(config=config, wind_farm='A')

model = FaultDetector(config=config, model_directory="../results")
train_results = model.fit(sensor_data=train_sensor_data, normal_index=train_status, save_models=True)

# save the model manually if needed by using
# model.save_models(model_name='my_model') or set save_models=True (default value)

2026-10-25 14:30:26,374 - energy_fault_detector - INFO - Initialize models..
2026-10-25 14:30:26,701 - energy_fault_detector - INFO - Fit preprocessor pipeline.
2026-10-25 14:30:27,254 - energy_fault_detector - INFO - Train autoencoder.
Epoch 1/1000
270/270 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.2295 - mean_absolute_error: 0.3249 - val_loss: 0.0918 - val_mean_absolute_error: 0.2153
Epoch 2/1000
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0680 - mean_absolute_error: 0.1850 - val_loss: 0.0531 - val_mean_absolute_error: 0.1644
Epoch 3/1000
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0449 - mean_absolute_error: 0.1517 - val_loss: 0.0412 - val_mean_absolute_error: 0.1460
Epoch 4/1000
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0358 - mean_absolute_error: 0.1358 - val_loss: 0.0336 - val_mean_absolute_error: 0.1320
Epoch 5/1000
270/270 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0302 - mean_absolute_error: 0.1248 - val_loss: 0.0290 - val_mean_absolute_error: 0.1221
Epo

In [21]:
train_results.model_date, train_results.model_path

('20261025_143131',
 '/cluster/home/selmaka/SCADA_model_testing/results/20261025_143131')

The FaultDetector.predict method returns a FaultDetectionResult object containing several result dataframes. <br>

**recon_error**: reconstruction error of the autoencoder<br>
**reconstruction**: reconstruction of the input data (rescaled to original scale)<br>
**predicted_anomalies**: series indicating whether an anomaly is detected<br>
**anomaly_score**: dataframe with column containing the Anomaly Score<br>
**bias_data**: dataframe with ARCANA bias vector, if root cause analysis was performed<br>
**arcana_losses**: dataframe containing recorded ARCANA losses<br>
**tracked_bias**: List of recorded biases during ARCANA iterations<br>

In [24]:
predictions = model.predict(sensor_data=test_sensor_data)

scores = pd.DataFrame({
    "event_id": event_id,
    "time_stamp": predictions.anomaly_score.index,
    "anomaly_score": predictions.anomaly_score.values,
    "predicted_anomaly": predictions.predicted_anomalies.values,
})
scores.to_csv(f"{train_results.model_path}/scores.csv")
scores.head()

155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 622us/step


,event_id,time_stamp,anomaly_score,predicted_anomaly
0,40,2022-12-25 00:00:00,0.831964,False
1,40,2022-12-25 00:10:00,0.672310,False
2,40,2022-12-25 00:20:00,0.710605,False
3,40,2022-12-25 00:30:00,0.961502,False
4,40,2022-12-25 00:40:00,0.779310,False


### Plot the evaluation results

In [ ]:
fig, ax = plot_learning_curve(model)

In [ ]:
fig, ax = plot_score_with_threshold(model=model, data=test_sensor_data, normal_index=test_status)
ax.set_yscale('log')

In [ ]:
# Add the anomaly event
fig, ax = plot_score_with_threshold(model=model, data=test_sensor_data, normal_index=test_status)
ax.set_yscale('log')
ax.axvspan(event_info['event_start'], event_info['event_end'], alpha=0.1, color='red', label='anomaly event')
ax.legend(loc='upper left')

In [ ]:
# Add criticality information
fig, ax = plot_score_with_threshold(model=model, data=test_sensor_data, normal_index=test_status, show_criticality=True, figsize=(15, 6))
ax.set_yscale('log')
ax.axvspan(event_info['event_start'], event_info['event_end'], alpha=0.1, color='red', label='anomaly event')
ax.legend(loc='upper left')

In [ ]:
# Plot reconstruction of the first couple of features
fig, ax = plot_reconstruction(test_sensor_data, predictions.reconstruction, features_to_plot=predictions.reconstruction.columns[:20])

In [ ]:
# Plot the feature with the anomalous behaviour
fig, ax = plot_reconstruction(test_sensor_data, predictions.reconstruction, features_to_plot=['Temperature in the split ring chamber (avg)'])

### Evaluate results

In [12]:
from sklearn.metrics import fbeta_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# create boolean series with the ground truth for this dataset
ground_truth = CAREScore.create_ground_truth(
    event_start=event_info['event_start'],
    event_end=event_info['event_end'],
    event_label=event_info['event_label'],
    normal_index=test_status
)

predicted_anomalies = predictions.predicted_anomalies

fbeta_score(y_true=ground_truth, y_pred=predicted_anomalies, beta=1)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix(ground_truth, predicted_anomalies, labels=[False, True]),
                              display_labels=["Normal", "Anomaly"],)
disp.plot()